# LeetCode 1351 — Count Negative Numbers in a Sorted Matrix bằng Python

**Link đề:** https://leetcode.com/problems/count-negative-numbers-in-a-sorted-matrix/
**Độ khó:** Easy · **Chủ đề:** Array, Binary Search, Matrix

```python
class Solution:
    def countNegatives(self, grid: list[list[int]]) -> int:
        ...
```

## Mục tiêu học
1. Biết đọc/duyệt **ma trận** (list lồng list) trong Python: `grid[i][j]`, `len(grid)`, `len(grid[0])`.
2. Hiểu ý nghĩa "sắp xếp **không tăng** theo hàng và theo cột" và khai thác nó.
3. Đi qua 4 lời giải: duyệt toàn bộ → dừng sớm mỗi hàng → **binary search** mỗi hàng → **đi cầu thang** (staircase) `O(m + n)`.
4. Trace từng bước, chứng minh bằng invariant, phân tích Time/Space.
5. Tự giải các biến thể: đếm số `0`, đếm phần tử `< k`, tìm giá trị trong ma trận sắp xếp.

**Cách học:** đọc markdown → đoán output cell code kế tiếp → chạy → so sánh. Cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — đếm số âm trong ma trận đã sắp xếp

Cho ma trận `grid` kích thước `m x n`. Mỗi **hàng** và mỗi **cột** đều được sắp xếp theo thứ tự **không tăng** (non-increasing: phần tử sau `<=` phần tử trước). Hãy trả về **số lượng số âm** trong `grid`.

**Ví dụ 1**

```text
grid =
 4   3   2  -1
 3   2   1  -1
 1   1  -1  -2
-1  -1  -2  -3
```
Output: `8` (có 8 số âm).

**Ví dụ 2:** `grid = [[3,2],[1,0]]` → Output `0` (lưu ý: `0` **không** phải số âm).

**Constraints**
- `m == grid.length`, `n == grid[i].length`
- `1 <= m, n <= 100`
- `-100 <= grid[i][j] <= 100`

**Follow up:** Có tìm được lời giải `O(n + m)` không?

Câu hỏi đặt ra trước khi code:
- Nếu bỏ qua chữ "sorted", bài này giải thế nào? (rất dễ, nhưng tốn bao nhiêu?)
- Thứ tự sắp xếp cho ta thông tin gì về **vị trí** của các số âm?

In [1]:
grid1 = [[4, 3, 2, -1],
         [3, 2, 1, -1],
         [1, 1, -1, -2],
         [-1, -1, -2, -3]]
grid2 = [[3, 2], [1, 0]]

for row in grid1:
    print(row)
print("Số hàng m =", len(grid1), "| số cột n =", len(grid1[0]))

[4, 3, 2, -1]
[3, 2, 1, -1]
[1, 1, -1, -2]
[-1, -1, -2, -3]
Số hàng m = 4 | số cột n = 4


## 2. Nền tảng Python: ma trận là list lồng list

Trong Python, ma trận thường được lưu là **list của các list**:
- `grid[i]` là **hàng thứ i** (một list).
- `grid[i][j]` là phần tử ở **hàng i, cột j** (đếm từ 0).
- `len(grid)` = số hàng `m`; `len(grid[0])` = số cột `n` (mọi hàng dài bằng nhau).

Lưu ý thứ tự: **hàng trước, cột sau**. `grid[0][3]` là hàng 0, cột 3.

In [2]:
g = [[4, 3, 2, -1],
     [3, 2, 1, -1]]
print("Hàng 0:", g[0])
print("g[0][3] =", g[0][3], "(hàng 0, cột 3)")
print("g[1][0] =", g[1][0], "(hàng 1, cột 0)")
print("Phần tử cuối hàng 1:", g[1][-1])   # chỉ số âm -1 = phần tử cuối

Hàng 0: [4, 3, 2, -1]
g[0][3] = -1 (hàng 0, cột 3)
g[1][0] = 3 (hàng 1, cột 0)
Phần tử cuối hàng 1: -1


### 2.1. Hai cách duyệt mọi phần tử

1. Duyệt bằng **chỉ số**: `for i in range(m): for j in range(n): ...` — dùng khi cần biết vị trí.
2. Duyệt bằng **giá trị**: `for row in grid: for x in row: ...` — gọn khi chỉ cần giá trị.

`range(m)` sinh dãy `0, 1, ..., m-1` (không gồm `m`).

In [3]:
g = [[3, 2], [1, 0]]
print("Duyệt theo chỉ số:")
for i in range(len(g)):
    for j in range(len(g[0])):
        print(f"  g[{i}][{j}] = {g[i][j]}")

print("Duyệt theo giá trị:")
for row in g:
    for x in row:
        print("  x =", x)

Duyệt theo chỉ số:
  g[0][0] = 3
  g[0][1] = 2
  g[1][0] = 1
  g[1][1] = 0
Duyệt theo giá trị:
  x = 3
  x = 2
  x = 1
  x = 0


### 2.2. `sum` với biểu thức sinh (generator expression)

`sum(1 for x in row if x < 0)` đọc là: "với mỗi `x` trong `row`, nếu `x < 0` thì sinh ra số `1`; cộng tất cả lại". Đây là cách đếm gọn của Python.

Mẹo nữa: trong Python `True` cư xử như `1`, `False` như `0`, nên `sum(x < 0 for x in row)` cũng đếm được.

In [4]:
row = [4, 3, -1, -2]
print(sum(1 for x in row if x < 0))     # 2
print(sum(x < 0 for x in row))          # 2 — True được cộng như 1
print(True + True + False)              # 2

2
2
2


### 2.3. "Không tăng" nghĩa là gì?

Dãy **không tăng**: `a[0] >= a[1] >= a[2] >= ...` (cho phép bằng nhau). Ví dụ `[4, 3, 3, -1]` là không tăng.

Hệ quả quan trọng trong một hàng: **một khi gặp số âm, mọi số phía sau cũng âm** (vì chúng còn nhỏ hơn hoặc bằng). Vậy các số âm của một hàng luôn nằm thành **một đoạn liền ở cuối hàng**.

Tương tự với cột: các số âm của một cột nằm thành một đoạn liền ở **dưới cùng** của cột.

In [5]:
def is_non_increasing(a):
    # all(...) trả True nếu MỌI điều kiện đều đúng
    return all(a[k] >= a[k + 1] for k in range(len(a) - 1))

print(is_non_increasing([4, 3, 3, -1]))   # True
print(is_non_increasing([4, 5, 3]))       # False

row = [4, 3, 2, -1, -1, -5]
first_neg = next(k for k, x in enumerate(row) if x < 0)
print("Số âm đầu tiên ở cột", first_neg, "→ đoạn âm:", row[first_neg:])

True
False
Số âm đầu tiên ở cột 3 → đoạn âm: [-1, -1, -5]


Ở cell trên có hai công cụ mới:
- `enumerate(row)` sinh cặp `(chỉ số, giá trị)`: `(0,4), (1,3), ...`.
- `next(generator)` lấy **phần tử đầu tiên** mà generator sinh ra — ở đây là chỉ số của số âm đầu tiên.
- `row[first_neg:]` là **slicing**: lấy từ vị trí `first_neg` đến hết.

Nếu một hàng có `n` phần tử và số âm đầu tiên ở cột `k` thì hàng đó có **`n - k` số âm**. Công thức này là trái tim của các cách tối ưu.

In [6]:
row = [4, 3, 2, -1, -1, -5]
n = len(row)
k = 3
print("n - k =", n - k, "| đếm trực tiếp =", sum(x < 0 for x in row))

n - k = 3 | đếm trực tiếp = 3


## 3. Cách 1 — Duyệt toàn bộ (Brute Force)

Ý tưởng tự nhiên nhất: đi qua **mọi ô**, ô nào `< 0` thì tăng biến đếm. Cách này **không dùng** thông tin "đã sắp xếp" chút nào.

In [7]:
def count_negatives_brute(grid):
    count = 0
    for row in grid:
        for x in row:
            if x < 0:
                count += 1
    return count

print(count_negatives_brute(grid1))   # 8
print(count_negatives_brute(grid2))   # 0

8
0


### Độ phức tạp của Brute Force

- Thời gian: `O(m·n)` — xem mọi ô đúng một lần. Với `m = n = 100` là 10.000 ô, vẫn rất nhanh.
- Bộ nhớ: `O(1)`.

Cách này **đã đủ để Accepted**. Nhưng follow-up hỏi `O(m + n)`, và bài là bài luyện "khai thác tính sắp xếp", nên ta đi tiếp.

**Nút thắt:** khi đang ở một hàng và đã gặp số âm đầu tiên, ta vẫn tiếp tục xem các ô còn lại — dù **chắc chắn** chúng âm. Đó là công việc thừa.

## 4. Cách 2 — Dừng sớm trong mỗi hàng

Từ mục 2.3: trong một hàng, khi gặp số âm đầu tiên ở cột `j`, hàng đó có `n - j` số âm → cộng luôn và `break` (thoát vòng `for` bên trong).

`break` chỉ thoát **vòng lặp gần nhất** chứa nó, vòng ngoài vẫn chạy tiếp.

In [8]:
for i in range(3):
    for j in range(5):
        if j == 2:
            break          # chỉ thoát vòng j
        print(f"i={i}, j={j}")
print("vòng ngoài chạy đủ 3 lần")

i=0, j=0
i=0, j=1
i=1, j=0
i=1, j=1
i=2, j=0
i=2, j=1
vòng ngoài chạy đủ 3 lần


In [9]:
def count_negatives_early_stop(grid):
    n = len(grid[0])
    count = 0
    for row in grid:
        for j in range(n):
            if row[j] < 0:
                count += n - j     # từ j tới cuối hàng đều âm
                break
    return count

print(count_negatives_early_stop(grid1))   # 8
print(count_negatives_early_stop(grid2))   # 0

8
0


Cách này nhanh hơn khi số âm nhiều, nhưng trường hợp xấu nhất (ma trận toàn số dương) vẫn phải xem mọi ô → vẫn `O(m·n)`.

Muốn **chắc chắn** nhanh hơn, ta cần tìm "số âm đầu tiên" mà **không** đi từng ô. Đây là chỗ binary search xuất hiện.

## 5. Cách 3 — Binary Search trên mỗi hàng

### 5.1. Binary search (tìm kiếm nhị phân) là gì?

Trên một dãy đã sắp xếp, thay vì xem từng phần tử, ta xem **phần tử giữa** và loại bỏ **một nửa** dãy mỗi lần. Dãy dài `n` chỉ cần khoảng `log2(n)` bước.

Bài toán con của ta: trong hàng không tăng, tìm **chỉ số nhỏ nhất `k` sao cho `row[k] < 0`** (nếu không có thì `k = n`).

Hàng có dạng: `[không âm, không âm, ..., âm, âm]` — điều kiện `row[mid] < 0` có dạng `False False ... True True`. Ta tìm vị trí `True` đầu tiên.

In [10]:
row = [4, 3, 2, -1, -1, -5]
print([x < 0 for x in row])   # False ... True — "ranh giới" nằm ở chỉ số 3

[False, False, False, True, True, True]


### 5.2. Viết binary search tìm ranh giới

Giữ khoảng tìm `[lo, hi)` (gồm `lo`, không gồm `hi`):
- `lo = 0`, `hi = n` (đáp án có thể là `n` khi không có số âm).
- Lặp khi `lo < hi`: `mid = (lo + hi) // 2`.
  - Nếu `row[mid] < 0`: `mid` có thể là đáp án, nhưng còn có thể có số âm sớm hơn → `hi = mid`.
  - Ngược lại `row[mid] >= 0`: đáp án nằm sau `mid` → `lo = mid + 1`.
- Kết thúc: `lo == hi` chính là chỉ số âm đầu tiên.

`//` là chia lấy phần nguyên: `7 // 2 = 3`.

In [11]:
def first_negative_index(row):
    lo, hi = 0, len(row)
    while lo < hi:
        mid = (lo + hi) // 2
        if row[mid] < 0:
            hi = mid
        else:
            lo = mid + 1
    return lo

for r in [[4, 3, 2, -1, -1, -5], [3, 2], [-1, -2], [1, 0, -1], [0]]:
    k = first_negative_index(r)
    print(r, "→ k =", k, "| số âm =", len(r) - k)

[4, 3, 2, -1, -1, -5] → k = 3 | số âm = 3
[3, 2] → k = 2 | số âm = 0
[-1, -2] → k = 0 | số âm = 2
[1, 0, -1] → k = 2 | số âm = 1
[0] → k = 1 | số âm = 0


### 5.3. Trace binary search trên một hàng

Hàng `[4, 3, 2, -1, -1, -5]`:

| Bước | lo | hi | mid | row[mid] | Quyết định |
|---|---|---|---|---|---|
| 1 | 0 | 6 | 3 | -1 | âm → hi = 3 |
| 2 | 0 | 3 | 1 | 3 | không âm → lo = 2 |
| 3 | 2 | 3 | 2 | 2 | không âm → lo = 3 |
| hết | 3 | 3 | | | trả 3 → 6 - 3 = 3 số âm |

In [12]:
def trace_first_negative(row):
    lo, hi = 0, len(row)
    step = 1
    while lo < hi:
        mid = (lo + hi) // 2
        if row[mid] < 0:
            action = f"âm → hi = {mid}"
            hi = mid
        else:
            action = f"không âm → lo = {mid + 1}"
            lo = mid + 1
        print(f"Bước {step}: mid={mid}, row[mid]={row[mid]:>3} | {action} | khoảng [{lo},{hi})")
        step += 1
    print("Chỉ số âm đầu tiên:", lo)
    return lo

trace_first_negative([4, 3, 2, -1, -1, -5])

Bước 1: mid=3, row[mid]= -1 | âm → hi = 3 | khoảng [0,3)
Bước 2: mid=1, row[mid]=  3 | không âm → lo = 2 | khoảng [2,3)
Bước 3: mid=2, row[mid]=  2 | không âm → lo = 3 | khoảng [3,3)
Chỉ số âm đầu tiên: 3


3

In [13]:
def count_negatives_binary(grid):
    n = len(grid[0])
    total = 0
    for row in grid:
        total += n - first_negative_index(row)
    return total

print(count_negatives_binary(grid1))   # 8
print(count_negatives_binary(grid2))   # 0

8
0


### 5.4. Dùng thư viện `bisect` (tuỳ chọn)

Module chuẩn `bisect` có sẵn binary search, nhưng nó giả định dãy **tăng dần**. Hàng của ta giảm dần → mẹo: đổi dấu từng phần tử (`-x`) thì dãy thành tăng dần, và "x < 0" ⇔ "-x > 0". `bisect_right(a, 0)` trả về số phần tử `<= 0` trong `a` tăng dần — chính là chỉ số đầu tiên có `-x > 0`.

Từ Python 3.10, `bisect` còn nhận tham số `key=`; để tương thích, ở đây ta tạo list đổi dấu (tốn `O(n)` cho mỗi hàng nên **không** còn lợi về Big-O, chỉ để minh hoạ).

In [14]:
from bisect import bisect_right

row = [4, 3, 2, -1, -1, -5]
neg = [-x for x in row]          # [-4, -3, -2, 1, 1, 5] tăng dần
k = bisect_right(neg, 0)
print(neg, "→ k =", k, "| số âm =", len(row) - k)

[-4, -3, -2, 1, 1, 5] → k = 3 | số âm = 3


### Độ phức tạp Cách 3

- Thời gian: `O(m · log n)` — mỗi hàng một binary search.
- Bộ nhớ: `O(1)` (bản tự viết).

Tốt hơn `O(m·n)`, nhưng vẫn **chưa dùng** tính sắp xếp theo **cột**. Follow-up yêu cầu `O(m + n)`.

## 6. Bước đột phá — ranh giới âm/không âm là một "cầu thang"

Gọi `k[i]` = chỉ số âm đầu tiên của hàng `i`. Quan sát ví dụ 1:

| Hàng i | Hàng | k[i] |
|---|---|---|
| 0 | `4 3 2 -1` | 3 |
| 1 | `3 2 1 -1` | 3 |
| 2 | `1 1 -1 -2` | 2 |
| 3 | `-1 -1 -2 -3` | 0 |

`k[i]` **không tăng** khi đi xuống! Lý do: vì cột cũng không tăng, `grid[i+1][j] <= grid[i][j]`. Nếu `grid[i][j] < 0` thì ô ngay dưới cũng `< 0`. Vậy số âm đầu tiên của hàng dưới **không thể nằm bên phải** số âm đầu tiên của hàng trên: `k[i+1] <= k[i]`.

**Hệ quả:** khi xuống hàng mới, ta không cần tìm lại từ đầu — chỉ cần **tiếp tục đi sang trái** từ vị trí cũ. Con trỏ cột chỉ đi sang trái tổng cộng tối đa `n` bước, con trỏ hàng đi xuống `m` bước → `O(m + n)`.

In [15]:
ks = [first_negative_index(row) for row in grid1]
print("k theo từng hàng:", ks)
print("k không tăng?", all(ks[i] >= ks[i + 1] for i in range(len(ks) - 1)))

k theo từng hàng: [3, 3, 2, 0]
k không tăng? True


## 7. Cách 4 — Staircase (đi cầu thang) `O(m + n)`

Bắt đầu ở **góc dưới-trái** `(i = m-1, j = 0)`, tức hàng chứa nhiều số âm nhất:
- Nếu `grid[i][j] < 0`: mọi ô từ cột `j` tới cuối hàng `i` đều âm → cộng `n - j`, rồi **đi lên** một hàng (`i -= 1`).
- Ngược lại (`>= 0`): ô này chưa âm → **sang phải** (`j += 1`) tìm số âm.
- Dừng khi `i < 0` (hết hàng) hoặc `j == n` (hết cột: các hàng còn lại không có số âm).

Vì sao đi từ dưới lên? Đi lên thì `k[i]` chỉ có thể **tăng** (sang phải), nên con trỏ `j` chỉ đi sang phải, không bao giờ lùi.

(Cũng có thể bắt đầu từ góc trên-phải và đi xuống/sang trái — đối xứng, xem phần biến thể.)

### 7.1. Mổ xẻ từng đoạn

```python
m, n = len(grid), len(grid[0])
i, j = m - 1, 0          # góc dưới-trái
count = 0
while i >= 0 and j < n:  # còn trong ma trận
    if grid[i][j] < 0:
        count += n - j   # cả đuôi hàng i là âm
        i -= 1           # hàng i xong, đi lên
    else:
        j += 1           # chưa âm, sang phải
```

Mỗi vòng lặp **hoặc** giảm `i` **hoặc** tăng `j` → tối đa `m + n` vòng.

In [16]:
def count_negatives_staircase(grid):
    m, n = len(grid), len(grid[0])
    i, j = m - 1, 0
    count = 0
    while i >= 0 and j < n:
        if grid[i][j] < 0:
            count += n - j
            i -= 1
        else:
            j += 1
    return count

print(count_negatives_staircase(grid1))   # 8
print(count_negatives_staircase(grid2))   # 0

8
0


## 8. TRACE từng bước — Staircase trên ví dụ 1

```text
       j=0 j=1 j=2 j=3
i=0     4   3   2  -1
i=1     3   2   1  -1
i=2     1   1  -1  -2
i=3    -1  -1  -2  -3
```

| Bước | i | j | grid[i][j] | Hành động | count |
|---|---|---|---|---|---|
| 1 | 3 | 0 | -1 | âm: +4, lên | 4 |
| 2 | 2 | 0 | 1 | không âm: sang phải | 4 |
| 3 | 2 | 1 | 1 | sang phải | 4 |
| 4 | 2 | 2 | -1 | âm: +2, lên | 6 |
| 5 | 1 | 2 | 1 | sang phải | 6 |
| 6 | 1 | 3 | -1 | âm: +1, lên | 7 |
| 7 | 0 | 3 | -1 | âm: +1, lên | 8 |
| dừng | -1 | 3 | | i < 0 | **8** |

Chỉ 7 bước thay vì xem 16 ô.

In [17]:
def trace_staircase(grid):
    m, n = len(grid), len(grid[0])
    i, j = m - 1, 0
    count = 0
    step = 1
    while i >= 0 and j < n:
        v = grid[i][j]
        if v < 0:
            count += n - j
            print(f"Bước {step}: (i={i}, j={j}) = {v:>3} | âm → +{n - j}, lên       | count = {count}")
            i -= 1
        else:
            print(f"Bước {step}: (i={i}, j={j}) = {v:>3} | không âm → sang phải | count = {count}")
            j += 1
        step += 1
    print(f"Dừng ở i={i}, j={j} → kết quả {count}")
    return count

trace_staircase(grid1)

Bước 1: (i=3, j=0) =  -1 | âm → +4, lên       | count = 4
Bước 2: (i=2, j=0) =   1 | không âm → sang phải | count = 4
Bước 3: (i=2, j=1) =   1 | không âm → sang phải | count = 4
Bước 4: (i=2, j=2) =  -1 | âm → +2, lên       | count = 6
Bước 5: (i=1, j=2) =   1 | không âm → sang phải | count = 6
Bước 6: (i=1, j=3) =  -1 | âm → +1, lên       | count = 7
Bước 7: (i=0, j=3) =  -1 | âm → +1, lên       | count = 8
Dừng ở i=-1, j=3 → kết quả 8


8

### 8.1. Trace thêm: ma trận không có số âm, và ma trận toàn âm

Hai trường hợp này cho thấy **hai điều kiện dừng** khác nhau: `j == n` (đi hết sang phải) và `i < 0` (đi hết lên trên).

In [18]:
trace_staircase([[3, 2], [1, 0]])
print()
trace_staircase([[-1, -2], [-3, -4]])

Bước 1: (i=1, j=0) =   1 | không âm → sang phải | count = 0
Bước 2: (i=1, j=1) =   0 | không âm → sang phải | count = 0
Dừng ở i=1, j=2 → kết quả 0

Bước 1: (i=1, j=0) =  -3 | âm → +2, lên       | count = 2
Bước 2: (i=0, j=0) =  -1 | âm → +2, lên       | count = 4
Dừng ở i=-1, j=0 → kết quả 4


4

## 9. Code nộp LeetCode

Bản Staircase `O(m + n)` (đáp ứng follow-up):

In [19]:
class Solution:
    def countNegatives(self, grid: list[list[int]]) -> int:
        m, n = len(grid), len(grid[0])
        i, j = m - 1, 0
        count = 0
        while i >= 0 and j < n:
            if grid[i][j] < 0:
                count += n - j
                i -= 1
            else:
                j += 1
        return count

sol = Solution()
print(sol.countNegatives([[4,3,2,-1],[3,2,1,-1],[1,1,-1,-2],[-1,-1,-2,-3]]))  # 8
print(sol.countNegatives([[3,2],[1,0]]))                                        # 0

8
0


Bản Binary Search mỗi hàng (cũng Accepted, `O(m log n)`):

In [20]:
class SolutionBinary:
    def countNegatives(self, grid: list[list[int]]) -> int:
        n = len(grid[0])
        total = 0
        for row in grid:
            lo, hi = 0, n
            while lo < hi:
                mid = (lo + hi) // 2
                if row[mid] < 0:
                    hi = mid
                else:
                    lo = mid + 1
            total += n - lo
        return total

print(SolutionBinary().countNegatives([[4,3,2,-1],[3,2,1,-1],[1,1,-1,-2],[-1,-1,-2,-3]]))  # 8

8


## 10. Vì sao Staircase đúng?

**Invariant (bất biến)** — trước mỗi vòng lặp:
1. Mọi hàng `> i` (bên dưới) đã được đếm đầy đủ vào `count`.
2. Với hàng `i` hiện tại, mọi ô ở cột `< j` đều **không âm**.

Kiểm tra:
- Ban đầu `i = m-1`, `j = 0`: không có hàng nào bên dưới, không có cột nào `< 0` → đúng.
- Nhánh `grid[i][j] >= 0`: cột `j` của hàng `i` không âm → tăng `j` vẫn giữ (2).
- Nhánh `grid[i][j] < 0`: theo (2) và tính sắp xếp hàng, hàng `i` có **đúng** `n - j` số âm → cộng vào, hàng `i` xong → giữ (1). Khi lên hàng `i-1`: với mọi cột `c < j`, `grid[i-1][c] >= grid[i][c] >= 0` (cột không tăng) → (2) vẫn đúng cho hàng mới mà **không cần lùi `j`**.
- Kết thúc: nếu `i < 0` thì mọi hàng đã đếm. Nếu `j == n` thì theo (2) hàng `i` không có số âm, và mọi hàng phía trên còn "lớn hơn" nên cũng không có → đếm đủ.

Mỗi vòng `i` giảm hoặc `j` tăng → vòng lặp chắc chắn dừng sau tối đa `m + n` bước.

## 11. Phân tích Time / Space

Gọi `m` = số hàng, `n` = số cột.

| Cách | Thời gian | Bộ nhớ | Ghi chú |
|---|---|---|---|
| 1. Duyệt toàn bộ | `O(m·n)` | `O(1)` | không dùng tính sắp xếp |
| 2. Dừng sớm mỗi hàng | `O(m·n)` worst | `O(1)` | nhanh khi nhiều số âm |
| 3. Binary search mỗi hàng | `O(m·log n)` | `O(1)` | chỉ dùng sắp xếp theo hàng |
| 4. Staircase | `O(m + n)` | `O(1)` | dùng cả hàng lẫn cột — tối ưu |

Với `m = n = 100`: cách 1 ~ 10.000 phép xem, cách 3 ~ 700, cách 4 ≤ 200. Đếm số bước thực tế ở cell dưới.

In [21]:
def steps_staircase(grid):
    m, n = len(grid), len(grid[0])
    i, j, steps = m - 1, 0, 0
    while i >= 0 and j < n:
        steps += 1
        if grid[i][j] < 0:
            i -= 1
        else:
            j += 1
    return steps

# Ma trận 100x100: grid[i][j] = 100 - i - j  (không tăng theo hàng và cột)
big = [[100 - i - j for j in range(100)] for i in range(100)]
print("Số ô:", 100 * 100)
print("Bước staircase:", steps_staircase(big))
print("Đáp án:", count_negatives_staircase(big), "=", count_negatives_brute(big))

Số ô: 10000
Bước staircase: 198
Đáp án: 4851 = 4851


Cell trên dùng **list comprehension lồng**: `[[biểu_thức for j in ...] for i in ...]` tạo ma trận. Vòng `for` bên ngoài (theo `i`) tạo từng hàng; vòng bên trong (theo `j`) tạo từng ô của hàng đó.

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

Nhóm test:
1. Ví dụ của đề.
2. Ma trận 1x1: âm / không âm / bằng 0.
3. Một hàng, một cột.
4. Toàn âm, toàn dương, có số 0 (0 không phải âm).
5. Ma trận ngẫu nhiên **hợp lệ** (đã sắp xếp) — đối chiếu 4 cách.

In [22]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
cases = [
    ("đề 1", [[4,3,2,-1],[3,2,1,-1],[1,1,-1,-2],[-1,-1,-2,-3]], 8),
    ("đề 2", [[3,2],[1,0]], 0),
    ("1x1 âm", [[-5]], 1),
    ("1x1 dương", [[5]], 0),
    ("1x1 bằng 0", [[0]], 0),
    ("một hàng", [[5, 1, 0, -1, -3]], 2),
    ("một cột", [[5], [0], [-1], [-2]], 2),
    ("toàn âm", [[-1, -2], [-3, -4]], 4),
    ("toàn số 0", [[0, 0], [0, 0]], 0),
    ("ranh giới lệch", [[1, -1], [-1, -1]], 3),
]
for name, g, exp in cases:
    check(name, sol.countNegatives(g), exp)

PASS đề 1: 8
PASS đề 2: 0
PASS 1x1 âm: 1
PASS 1x1 dương: 0
PASS 1x1 bằng 0: 0
PASS một hàng: 2
PASS một cột: 2
PASS toàn âm: 4
PASS toàn số 0: 0
PASS ranh giới lệch: 3


### 12.1. Sinh ma trận sắp xếp ngẫu nhiên để đối chiếu

Cách sinh: tạo ma trận ngẫu nhiên, sắp **mỗi hàng** giảm dần, rồi sắp **mỗi cột** giảm dần. Sắp cột sau khi sắp hàng vẫn giữ hàng được sắp (đây là một tính chất đã biết của việc sắp hàng-rồi-cột). Ta vẫn `assert` kiểm tra lại cho chắc.

`random.Random(1351)` tạo bộ sinh số ngẫu nhiên có **seed** cố định → mỗi lần chạy cho cùng kết quả.

In [23]:
import random

def random_sorted_grid(rng, m, n):
    g = [[rng.randint(-100, 100) for _ in range(n)] for _ in range(m)]
    for row in g:
        row.sort(reverse=True)                     # sắp hàng giảm dần
    for j in range(n):                             # sắp từng cột giảm dần
        col = sorted((g[i][j] for i in range(m)), reverse=True)
        for i in range(m):
            g[i][j] = col[i]
    return g

def is_valid(g):
    m, n = len(g), len(g[0])
    rows_ok = all(g[i][j] >= g[i][j + 1] for i in range(m) for j in range(n - 1))
    cols_ok = all(g[i][j] >= g[i + 1][j] for i in range(m - 1) for j in range(n))
    return rows_ok and cols_ok

rng = random.Random(1351)
for t in range(500):
    m, n = rng.randint(1, 8), rng.randint(1, 8)
    g = random_sorted_grid(rng, m, n)
    assert is_valid(g)
    expected = count_negatives_brute(g)
    assert count_negatives_early_stop(g) == expected
    assert count_negatives_binary(g) == expected
    assert count_negatives_staircase(g) == expected
    assert sol.countNegatives(g) == expected
print("500 ma trận ngẫu nhiên: 4 cách cho cùng kết quả — PASS")

500 ma trận ngẫu nhiên: 4 cách cho cùng kết quả — PASS


## 13. Lỗi người mới hay gặp

### Lỗi 1: Coi `0` là số âm

```python
if grid[i][j] <= 0:   # SAI
```
`0` không âm. Ví dụ 2 có số `0` và đáp án là `0`.

In [24]:
def wrong_le_zero(grid):
    return sum(1 for row in grid for x in row if x <= 0)

print("Sai:", wrong_le_zero([[3, 2], [1, 0]]), "| Đúng:", count_negatives_brute([[3, 2], [1, 0]]))

Sai: 1 | Đúng: 0


### Lỗi 2: Bắt đầu staircase ở góc sai

Bắt đầu ở **góc trên-trái** `(0, 0)`: ô lớn nhất. Nếu nó không âm, đi phải hay đi xuống đều có thể gặp số âm — không có quyết định "chắc chắn" nào, thuật toán bị kẹt. Góc đúng là góc mà **hai hướng đi có ý nghĩa ngược nhau**: dưới-trái (lên = lớn hơn, phải = nhỏ hơn) hoặc trên-phải.

Đoạn code sai dưới đây chỉ đi sang phải rồi xuống, bỏ sót số âm ở các hàng dưới mà nằm bên trái:

In [25]:
def wrong_top_left(grid):
    m, n = len(grid), len(grid[0])
    i, j, count = 0, 0, 0
    while i < m and j < n:
        if grid[i][j] < 0:
            count += n - j
            i += 1          # xuống hàng — nhưng không lùi j về trái!
        else:
            j += 1
    return count

g = [[4,3,2,-1],[3,2,1,-1],[1,1,-1,-2],[-1,-1,-2,-3]]
print("Sai:", wrong_top_left(g), "| Đúng:", count_negatives_staircase(g))

Sai: 4 | Đúng: 8


Đi từ trên xuống, `k[i]` giảm (dịch sang trái) — con trỏ `j` cần **lùi**, nên code trên sai. Sửa: bắt đầu dưới-trái như mục 7 (hoặc trên-phải và đi sang trái).

### Lỗi 3: Binary search sai biên

```python
lo, hi = 0, len(row) - 1   # SAI với khoảng [lo, hi)
```
Khi hàng **không có số âm**, đáp án phải là `n` nhưng `hi` bắt đầu từ `n-1` nên không bao giờ trả được `n` → đếm thừa 1.

In [26]:
def wrong_first_negative(row):
    lo, hi = 0, len(row) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if row[mid] < 0:
            hi = mid
        else:
            lo = mid + 1
    return lo

r = [3, 2, 1]
print("Sai: số âm =", len(r) - wrong_first_negative(r), "| Đúng:", len(r) - first_negative_index(r))

Sai: số âm = 1 | Đúng: 0


### Lỗi 4: Nhầm `len(grid)` với `len(grid[0])`

`len(grid)` là số **hàng**, `len(grid[0])` là số **cột**. Với ma trận không vuông (ví dụ 2x5), nhầm hai giá trị dẫn đến `IndexError` hoặc đếm sai. Luôn test ma trận **không vuông**.

In [27]:
g = [[1, 0, -1, -2, -3], [0, -1, -2, -3, -4]]   # 2 hàng, 5 cột
print("m =", len(g), "n =", len(g[0]), "→", count_negatives_staircase(g))
assert count_negatives_staircase(g) == count_negatives_brute(g) == 7

m = 2 n = 5 → 7


### Mẹo nhận dạng pattern

- **"Ma trận sắp xếp theo cả hàng và cột"** → nghĩ ngay tới **staircase** từ một góc mà hai hướng đi có ý nghĩa ngược nhau (dưới-trái hoặc trên-phải). Độ phức tạp `O(m + n)`.
- **"Dãy sắp xếp + tìm vị trí đầu tiên thỏa điều kiện"** → binary search tìm ranh giới `False...False True...True`.
- **"Đếm phần tử thỏa điều kiện đơn điệu"** → tìm ranh giới rồi lấy `n - k`, không cần đếm từng phần tử.
- Bài anh em: *Search a 2D Matrix II* (LeetCode 240), *Kth Smallest Element in a Sorted Matrix* (378), *The K Weakest Rows in a Matrix* (1337).

## 14. Biến thể

### 14.1. Staircase từ góc trên-phải

Bắt đầu `(0, n-1)`. Nếu ô âm → cả **cột** từ hàng `i` xuống dưới đều âm → cộng `m - i`, sang trái. Nếu không âm → đi xuống.

In [28]:
def count_negatives_top_right(grid):
    m, n = len(grid), len(grid[0])
    i, j, count = 0, n - 1, 0
    while i < m and j >= 0:
        if grid[i][j] < 0:
            count += m - i     # đếm theo cột
            j -= 1
        else:
            i += 1
    return count

print(count_negatives_top_right(grid1), count_negatives_top_right(grid2))   # 8 0

8 0


### 14.2. Đếm phần tử nhỏ hơn `k` bất kỳ

Thay điều kiện `< 0` bằng `< k`: thuật toán giữ nguyên vì điều kiện `x < k` vẫn **đơn điệu** theo hàng và cột.

In [29]:
def count_less_than(grid, k):
    m, n = len(grid), len(grid[0])
    i, j, count = m - 1, 0, 0
    while i >= 0 and j < n:
        if grid[i][j] < k:
            count += n - j
            i -= 1
        else:
            j += 1
    return count

print(count_less_than(grid1, 2))   # đếm các số < 2
print(sum(x < 2 for row in grid1 for x in row))

11
11


### 14.3. Tìm một giá trị trong ma trận sắp xếp (ý tưởng của LeetCode 240)

Cũng đi cầu thang từ góc dưới-trái. Với ma trận **không tăng** của bài này, ô dưới-trái là **lớn nhất của hàng** và **nhỏ nhất của cột**:
- Ô `< target`: cả hàng (bên phải còn nhỏ hơn) không thể chứa target → bỏ hàng, đi lên.
- Ô `> target`: cả cột (phía trên còn lớn hơn) không thể chứa target → bỏ cột, sang phải.

In [30]:
def contains(grid, target):
    m, n = len(grid), len(grid[0])
    i, j = m - 1, 0
    while i >= 0 and j < n:
        v = grid[i][j]
        if v == target:
            return True
        if v < target:
            i -= 1     # cần số lớn hơn: đi lên (cột không tăng → phía trên lớn hơn)
        else:
            j += 1     # cần số nhỏ hơn: sang phải
    return False

print(contains(grid1, -2), contains(grid1, 5), contains(grid1, 1))   # True False True

True False True


## 15. Bài tập (TODO — tự làm trước, đáp án ở cuối)

### Bài 1 — Tính tay
Với `grid = [[5, 1, -1], [2, -1, -2], [-1, -3, -4]]`, hãy tự lập bảng trace staircase (i, j, giá trị, hành động, count). Có bao nhiêu bước? Đáp số bao nhiêu?

### Bài 2 — Đếm số 0
Cho ma trận không tăng theo hàng và cột, đếm số ô **bằng 0** trong `O(m + n)`. (Gợi ý: số ô bằng 0 = số ô `< 1` trừ số ô `< 0`.)

In [31]:
def exercise_count_zeros(grid):
    # TODO: dùng hai lần đếm "nhỏ hơn k"
    pass

# Gợi ý test:
# assert exercise_count_zeros([[3, 0, 0], [0, 0, -1], [0, -1, -2]]) == 5

### Bài 3 — Viết lại binary search
Viết `exercise_first_negative(row)` trả chỉ số âm đầu tiên của hàng không tăng (hoặc `len(row)` nếu không có), **không** dùng `bisect`.

In [32]:
def exercise_first_negative(row):
    # TODO: lo, hi = 0, len(row); thu hẹp khoảng [lo, hi)
    pass

# Gợi ý test:
# assert exercise_first_negative([4, 3, -1]) == 2
# assert exercise_first_negative([1, 0]) == 2
# assert exercise_first_negative([-1]) == 0

### Bài 4 — Số âm trên mỗi hàng
Trả về list `res` với `res[i]` là số số âm của hàng `i`, tổng thời gian `O(m + n)`.

In [33]:
def exercise_negatives_per_row(grid):
    # TODO: staircase, ghi lại n - j cho từng hàng
    pass

# Gợi ý test:
# assert exercise_negatives_per_row([[4,3,2,-1],[3,2,1,-1],[1,1,-1,-2],[-1,-1,-2,-3]]) == [1, 1, 2, 4]

### Bài 5 — Ma trận sắp xếp **tăng dần** (non-decreasing)
Đếm số âm khi hàng và cột đều **không giảm**. Bắt đầu từ góc nào?

In [34]:
def exercise_count_negatives_increasing(grid):
    # TODO: số âm giờ nằm ở góc trên-trái
    pass

# Gợi ý test:
# assert exercise_count_negatives_increasing([[-3, -1, 2], [-2, 0, 3], [1, 2, 4]]) == 3

### Bài 6 — Mini challenge: hàng có nhiều số âm nhất
Trả về **chỉ số hàng** có nhiều số âm nhất (nếu hòa, lấy hàng nhỏ nhất). Có cần duyệt hết không? (Gợi ý: cột không tăng → hàng dưới luôn có ít nhất bằng số âm của hàng trên.)

In [35]:
def exercise_row_with_most_negatives(grid):
    # TODO
    pass

# Gợi ý test:
# assert exercise_row_with_most_negatives([[4,3,2,-1],[3,2,1,-1],[1,1,-1,-2],[-1,-1,-2,-3]]) == 3
# assert exercise_row_with_most_negatives([[-1, -1], [-1, -2]]) == 0

## 16. Gợi ý 3 mức

**Bài 2** — (1) Đếm "< 1" và "< 0" đều là bài cũ. (2) Dùng `count_less_than` với hai giá trị `k`. (3) `return count_less_than(grid, 1) - count_less_than(grid, 0)`.

**Bài 3** — (1) Điều kiện `row[mid] < 0` có dạng False…True. (2) `hi = len(row)`, nếu âm thì `hi = mid` ngược lại `lo = mid + 1`. (3) `while lo < hi: ...; return lo`.

**Bài 4** — (1) Mỗi lần staircase gặp ô âm là lúc hàng `i` được "chốt". (2) Tạo `res = [0] * m`, gán `res[i] = n - j` trước khi `i -= 1`. (3) Hàng nào không bao giờ được chốt (khi `j == n`) giữ giá trị `0`.

**Bài 5** — (1) Số âm nằm ở góc trên-trái, mỗi hàng có đoạn âm ở **đầu** hàng. (2) Bắt đầu từ góc trên-phải `(0, n-1)`: nếu âm thì cả đoạn `0..j` của hàng âm → cộng `j + 1`, đi xuống; nếu không thì sang trái. (3) `while i < m and j >= 0`.

**Bài 6** — (1) Số âm theo hàng không giảm khi đi xuống. (2) Hàng cuối luôn có nhiều nhất; nhưng hòa thì cần hàng nhỏ nhất. (3) Đếm số âm hàng cuối `c`, rồi tìm hàng đầu tiên từ trên xuống có số âm bằng `c` (dùng `exercise_negatives_per_row`).

## 17. Đáp án tham khảo (đặt cuối để không lộ trước khi làm)

**Bài 1:** trace:

| Bước | i | j | v | Hành động | count |
|---|---|---|---|---|---|
| 1 | 2 | 0 | -1 | +3, lên | 3 |
| 2 | 1 | 0 | 2 | phải | 3 |
| 3 | 1 | 1 | -1 | +2, lên | 5 |
| 4 | 0 | 1 | 1 | phải | 5 |
| 5 | 0 | 2 | -1 | +1, lên | 6 |

5 bước, đáp số **6**. Cell dưới kiểm tra lại.

In [36]:
assert trace_staircase([[5, 1, -1], [2, -1, -2], [-1, -3, -4]]) == 6

Bước 1: (i=2, j=0) =  -1 | âm → +3, lên       | count = 3
Bước 2: (i=1, j=0) =   2 | không âm → sang phải | count = 3
Bước 3: (i=1, j=1) =  -1 | âm → +2, lên       | count = 5
Bước 4: (i=0, j=1) =   1 | không âm → sang phải | count = 5
Bước 5: (i=0, j=2) =  -1 | âm → +1, lên       | count = 6
Dừng ở i=-1, j=2 → kết quả 6


In [37]:
def answer_count_zeros(grid):
    return count_less_than(grid, 1) - count_less_than(grid, 0)

def answer_first_negative(row):
    lo, hi = 0, len(row)
    while lo < hi:
        mid = (lo + hi) // 2
        if row[mid] < 0:
            hi = mid
        else:
            lo = mid + 1
    return lo

def answer_negatives_per_row(grid):
    m, n = len(grid), len(grid[0])
    res = [0] * m
    i, j = m - 1, 0
    while i >= 0 and j < n:
        if grid[i][j] < 0:
            res[i] = n - j
            i -= 1
        else:
            j += 1
    return res

def answer_count_negatives_increasing(grid):
    m, n = len(grid), len(grid[0])
    i, j, count = 0, n - 1, 0
    while i < m and j >= 0:
        if grid[i][j] < 0:
            count += j + 1
            i += 1
        else:
            j -= 1
    return count

def answer_row_with_most_negatives(grid):
    per_row = answer_negatives_per_row(grid)
    best = per_row[-1]
    for i, c in enumerate(per_row):
        if c == best:
            return i

assert answer_count_zeros([[3, 0, 0], [0, 0, -1], [0, -1, -2]]) == 5
assert answer_first_negative([4, 3, -1]) == 2
assert answer_first_negative([1, 0]) == 2
assert answer_first_negative([-1]) == 0
assert answer_negatives_per_row(grid1) == [1, 1, 2, 4]
assert answer_count_negatives_increasing([[-3, -1, 2], [-2, 0, 3], [1, 2, 4]]) == 3
assert answer_row_with_most_negatives(grid1) == 3
assert answer_row_with_most_negatives([[-1, -1], [-1, -2]]) == 0

# Đối chiếu ngẫu nhiên cho bài 2, 4, 5
rng = random.Random(7)
for _ in range(300):
    g = random_sorted_grid(rng, rng.randint(1, 6), rng.randint(1, 6))
    assert answer_count_zeros(g) == sum(x == 0 for r in g for x in r)
    assert answer_negatives_per_row(g) == [sum(x < 0 for x in r) for r in g]
    inc = [row[::-1] for row in g[::-1]]   # lật 2 chiều → tăng dần
    assert answer_count_negatives_increasing(inc) == count_negatives_brute(g)
print("Đáp án bài tập: PASS toàn bộ")

Đáp án bài tập: PASS toàn bộ


## 18. Tổng kết — template nhớ nhanh + checklist

```python
# Staircase trên ma trận không tăng theo hàng & cột
i, j = m - 1, 0                 # góc dưới-trái
while i >= 0 and j < n:
    if grid[i][j] < 0:          # điều kiện đơn điệu
        count += n - j          # cả đuôi hàng thỏa
        i -= 1
    else:
        j += 1
```

```python
# Binary search ranh giới False...True trên [lo, hi)
lo, hi = 0, n
while lo < hi:
    mid = (lo + hi) // 2
    if cond(mid): hi = mid
    else:         lo = mid + 1
return lo
```

**Checklist trước khi submit**
- [ ] `0` không bị tính là số âm (`< 0`, không phải `<= 0`).
- [ ] Bắt đầu staircase đúng góc (dưới-trái hoặc trên-phải).
- [ ] Điều kiện dừng `i >= 0 and j < n` (không truy cập ngoài ma trận).
- [ ] Binary search: `hi = len(row)` để xử lý hàng không có số âm.
- [ ] Đã test ma trận 1x1, một hàng, một cột, không vuông.